In [1]:
import sys
sys.path.append('..')
from src.commands.feature_extraction.main import Graph
from src.commands.feature_extraction.graph_utils import EdgeType

In [6]:
graph = Graph.load('/home/fellah-a/Documents/Text2KG/data/eval/health_care/graph')

In [5]:
graph._nodes['Expertise'].dataframe

,id,llm_label
0,0,Oncology Prognostic Biomarkers and Clinical Ou...
1,1,Immuno-Oncology and Targeted Therapy in Solid ...
2,2,Genetics of Neurodevelopmental Disorders and S...
3,3,Acute Myeloid Leukemia Pathogenesis and Therapy
4,4,Clinical Oncology and Hematology Outcomes Rese...
...,...,...
676,676,Food Allergen Detection and Management
677,677,Molecular Epidemiology and Antimicrobial Resis...
678,678,Photophysics of π-Expanded Diketopyrrolopyrrol...
679,679,Clinical Epidemiology of Atopic Dermatitis and...


In [28]:
graph._nodes['Triple'].dataframe.iloc[0]['id'] == graph._edges[EdgeType("Triple", "Article", "BELONGSTO")].dataframe.iloc[0]['src']

True

In [32]:
graph._nodes['Article']._nodes[0]

Node(id='5752895', properties={'id': '5752895', 'title': 'Impact of Tricuspid Regurgitation on Outcomes After Transcatheter Edge-to-Edge Repair for Primary Mitral Regurgitation: The PRIME-MR Registry', 'abstract': 'BACKGROUND : The presence of tricuspid regurgitation ( TR ) may affect prognosis in patients with primary mitral regurgitation . This study sought to investigate the impact of concomitant TR on clinical outcomes in patients with primary mitral regurgitation undergoing mitral transcatheter edge-to-edge repair ( M-TEER ) . METHODS : The PRIME-MR registry ( Outcomes of Patients Treated With Mitral Transcatheter Edge-to-edge Repair for Primary Mitral Regurgitation ) included 3083 patients with severe primary mitral regurgitation undergoing mitral transcatheter edge-to-edge repair ( M-TEER ) from 2008 to 2022 at 25 international sites . Only patients with successful mitral transcatheter edge-to-edge repair ( M-TEER ) procedures ( mitral regurgitation≤Moderate ) were analyzed . 30

In [30]:
graph._edges[EdgeType("Triple", "Article", "BELONGSTO")].dataframe

,id,src,dst
0,moderate_to_severe_tricuspid_regurgitation::af...,moderate_to_severe_tricuspid_regurgitation::af...,5752895
1,moderate_to_severe_tricuspid_regurgitation::im...,moderate_to_severe_tricuspid_regurgitation::im...,5752895
2,symptomatic_severe_primary_mitral_regurgitatio...,symptomatic_severe_primary_mitral_regurgitatio...,5752895
3,prime_mr_registry::include::symptomatic_severe...,prime_mr_registry::include::symptomatic_severe...,5752895
4,patient::underwent::m_teer_procedure-5752895,patient::underwent::m_teer_procedure,5752895
...,...,...,...
229127,secretor::susceptible_to::norovirus_infection-...,secretor::susceptible_to::norovirus_infection,2948761
229128,non_secretor::susceptible_to::infection-2948761,non_secretor::susceptible_to::infection,2948761
229129,non_secretor::susceptible_to::rotavirus-2948761,non_secretor::susceptible_to::rotavirus,2948761
229130,non_secretor::susceptible_to::norovirus_infect...,non_secretor::susceptible_to::norovirus_infection,2948761


In [27]:
import os

cache_path = os.path.join("/home/fellah-a/Documents/Text2KG/data/eval/health_care_v2/cache.jsonl")
cache = pd.read_json(cache_path, lines=True)
graph = Graph.load(os.path.join("/home/fellah-a/Documents/Text2KG/data/eval/health_care_v2/graph"))

In [34]:
cache[cache['task'] == 'align']['result'].apply(lambda x : x['result']['score']).mean()

np.float64(0.17342979780865242)

In [35]:
"""
Average DAS / CCS / CWAS computed straight from cache.jsonl.

Usage (from the project root):
    python cache_metrics.py <expertise_id>
"""
import os
import sys
from dataclasses import asdict

import numpy as np
import pandas as pd

from definitions import EMS_DIR, EVAL_DIR
from src.commands.feature_extraction.graph_utils import Graph, EdgeType


def to_dict(n) -> dict:
    d = asdict(n)
    p = d.pop('properties', {})
    return {**d, **p}


def main(expertise_id: str) -> None:

    # ---------------- Load cache ----------------
    raw = pd.read_json(cache_path, lines=True)
    df = raw.drop_duplicates(subset='task_id', keep='last').copy()
    df['ok'] = df['result'].apply(lambda x: isinstance(x, dict) and 'result' in x)

    print(f"Cache rows            : {len(raw)}  (unique tasks: {len(df)}, duplicates: {len(raw) - len(df)})")
    for task, g in df.groupby('task'):
        print(f"  {task:<6} tasks       : {len(g)}  (ok: {g['ok'].sum()}, errors: {(~g['ok']).sum()})")

    errors = df.loc[~df['ok'], 'result']
    if len(errors):
        names = errors.apply(lambda x: x.get('error', {}).get('name') if isinstance(x, dict) else None)
        print("  Error types         :", names.value_counts().to_dict())

    df = df[df['ok']].copy()
    df['score'] = df['result'].apply(lambda x: float(x['result']['score']))

    # ---------------- DAS ----------------
    # Set of (expertise, subdomain) relations that exist in the graph.
    graph_pairs = {
        (str(e.src), str(e.dst))
        for e in graph._edges[EdgeType(src="Expertise", dst="SubDomain", name="BELONGSTO")]._edges
    }

    align = df[df['task'] == 'align'][['expertise_id', 'domain_id', 'score']].copy()
    align['pair'] = list(zip(align['expertise_id'].astype(str), align['domain_id'].astype(str)))

    # Keep ONLY the align scores whose (expertise, subdomain) pair is an edge of the graph.
    in_graph = align['pair'].isin(graph_pairs)
    align_kept = align[in_graph]

    print(f"\nExpertise->SubDomain edges in graph : {len(graph_pairs)}")
    print(f"Align scores in cache               : {len(align)}")
    print(f"  kept (pair is a graph edge)       : {len(align_kept)}")
    print(f"  discarded (pair not in graph)     : {(~in_graph).sum()}")
    print(f"Graph edges with no align score     : {len(graph_pairs - set(align_kept['pair']))}")

    das = align_kept.groupby('expertise_id')['score'].mean().rename('DAS')

    # ---------------- CCS ----------------
    ccs = df[df['task'] == 'cover'].set_index('expertise_id')['score'].rename('CCS')

    # ---------------- CWAS ----------------
    scores = pd.concat([das, ccs], axis=1)
    scores['CWAS'] = np.sqrt(scores['DAS'] * scores['CCS'])

    n_exp = len(graph._nodes['Expertise']._nodes)
    print(f"\nExpertises in graph   : {n_exp}")
    print(f"  with DAS            : {scores['DAS'].notna().sum()}")
    print(f"  with CCS            : {scores['CCS'].notna().sum()}")
    print(f"  with all 3 metrics  : {scores.notna().all(axis=1).sum()}")

    print("\nAverages (over expertises that have the metric):")
    print(scores[['DAS', 'CCS', 'CWAS']].agg(['mean', 'std', 'min', 'max', 'count']).round(4).to_string())

    out = os.path.join(EVAL_DIR, expertise_id, 'scores.csv')
    scores.reset_index().to_csv(out, index=False)
    print(f"\nPer-expertise scores written to {out}")


main("")    

Cache rows            : 44946  (unique tasks: 44946, duplicates: 0)
  align  tasks       : 44265  (ok: 44265, errors: 0)
  cover  tasks       : 681  (ok: 681, errors: 0)

Expertise->SubDomain edges in graph : 1947
Align scores in cache               : 44265
  kept (pair is a graph edge)       : 1947
  discarded (pair not in graph)     : 42318
Graph edges with no align score     : 0

Expertises in graph   : 681
  with DAS            : 653
  with CCS            : 681
  with all 3 metrics  : 653

Averages (over expertises that have the metric):
            DAS       CCS      CWAS
mean     0.1712    0.5476    0.2575
std      0.1581    0.2319    0.1497
min      0.0000    0.0800    0.0000
max      0.9000    0.9500    0.7984
count  653.0000  681.0000  653.0000

Per-expertise scores written to /home/fellah-a/Documents/Text2KG/notebooks/../api/mount/data/eval/scores.csv


In [2]:
import sys
sys.path.append('..')
from src.manifest import Manifest

manifest = Manifest.load()

In [5]:
manifest.hf_models[0].exists()

True

In [1]:
import torch



In [2]:
import torch

def get_all_devices():
    
    devices = [torch.device("cpu")]

    # NVIDIA CUDA (also covers AMD ROCm builds)
    if torch.cuda.is_available():
        devices += [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]

    # Apple Silicon (Metal)
    if torch.backends.mps.is_available():
        devices.append(torch.device("mps"))

    # Intel GPUs
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        devices += [torch.device(f"xpu:{i}") for i in range(torch.xpu.device_count())]

    return devices

for d in get_all_devices():
    print(d)

cpu
cuda:0


In [14]:
import sys
sys.path.append('..')
from src.envirenment import env
from src.shared.platforms import PlatformsRegistry

In [18]:
gemini_platform = PlatformsRegistry.create("gemini")

In [19]:
result = await gemini_platform.get_models(api_key=env.get_api_key('gemini'))

In [20]:
result

<Success: models=['gemini-2.5-computer-use-preview-10-2025', 'gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-native-audio-latest', 'gemini-2.5-flash-native-audio-preview-09-2025', 'gemini-2.5-flash-native-audio-preview-12-2025', 'gemini-2.5-pro', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-4-argon', 'gemini-4-carbon', 'gemini-4-hydrogen', 'gemini-4-neon', 'gemini-decision-1', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-nano-banana-2.1', 'gemini-omni-1.1-flash', 'gemini-omni-1.2-flash', 'gemini-omni-flash-preview', 'gemini-pro-latest', 'gemini-robotics-er-2-preview', 'gemini-robotics-er-2-streaming-preview'] available=True>

In [1]:
import sys
sys.path.append('..')
from src.commands.feature_extraction.graph_utils import Graph, EdgeType, EdgeTypes

In [1]:
graph = Graph.load('/home/fellah-a/Documents/Text2KG/data/expertise_maps/nantes_uni_2020_2026_e5-v3')

NameError: name 'Graph' is not defined

In [4]:
graph.get_nodes('Article').nodes[0]

Node(id='5752895', properties={'id': '5752895', 'title': 'Impact of Tricuspid Regurgitation on Outcomes After Transcatheter Edge-to-Edge Repair for Primary Mitral Regurgitation: The PRIME-MR Registry', 'abstract': 'BACKGROUND : The presence of tricuspid regurgitation ( TR ) may affect prognosis in patients with primary mitral regurgitation . This study sought to investigate the impact of concomitant TR on clinical outcomes in patients with primary mitral regurgitation undergoing mitral transcatheter edge-to-edge repair ( M-TEER ) . METHODS : The PRIME-MR registry ( Outcomes of Patients Treated With Mitral Transcatheter Edge-to-edge Repair for Primary Mitral Regurgitation ) included 3083 patients with severe primary mitral regurgitation undergoing mitral transcatheter edge-to-edge repair ( M-TEER ) from 2008 to 2022 at 25 international sites . Only patients with successful mitral transcatheter edge-to-edge repair ( M-TEER ) procedures ( mitral regurgitation≤Moderate ) were analyzed . 30

In [ ]:
graph._edges[EdgeType("Entity", "Expertise", "BELONGSTO")].dataframe['dst'].nunique()

(37904, 3)